# Phylogenetic Distance via Data Smashing 2.0

This notebook demonstrates **lsmash** on real genomic data fetched live from NCBI.

We compare mitochondrial genomes from five species by encoding each DNA sequence as a
stream of integers (A→0, T→1, G→2, C→3) and computing pairwise Sequence Likelihood
divergences. The resulting distance matrix should reflect the known primate phylogenetic
tree — providing an interpretable ground-truth check on the algorithm.

**Expected result:** human ↔ chimp < human ↔ gorilla < human ↔ orangutan < human ↔ mouse

## 1. Fetch mitochondrial genomes from NCBI

In [ ]:
import urllib.request
import time

ORGANISMS = {
    'Human':     'NC_012920',
    'Chimp':     'NC_001643',
    'Gorilla':   'NC_011120',
    'Orangutan': 'NC_001646',
    'Mouse':     'NC_005089',
}

EFETCH = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'

def fetch_fasta(accession):
    url = f'{EFETCH}?db=nucleotide&id={accession}&rettype=fasta&retmode=text'
    with urllib.request.urlopen(url, timeout=30) as r:
        text = r.read().decode()
    lines = text.strip().splitlines()
    return ''.join(l for l in lines if not l.startswith('>')).upper()

ENCODE = {'A': 0, 'T': 1, 'G': 2, 'C': 3}

def encode_dna(seq):
    return [ENCODE[b] for b in seq if b in ENCODE]

labels, sequences = [], []
for name, acc in ORGANISMS.items():
    print(f'Fetching {name} ({acc})...', end=' ')
    raw = fetch_fasta(acc)
    enc = encode_dna(raw)
    labels.append(name)
    sequences.append(enc)
    print(f'{len(enc):,} symbols')
    time.sleep(0.4)  # respect NCBI rate limit

print('\nDone.')

## 2. Compute pairwise SL divergence with lsmash

In [ ]:
import lsmash as ls
import numpy as np

opt = ls.LsmashOptions()
opt.data_type = 'symbolic'
opt.data_dir  = 'row'
opt.sae        = True
opt.num_repeat = 20
opt.depth      = 8

print('Running lsmash...')
D = ls.from_sequences(sequences, opt)

print(f'Distance matrix ({D.shape[0]}x{D.shape[1]}):')
print(np.array2string(D, precision=4, suppress_small=True))

## 3. Visualise — heatmap and dendrogram

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Heatmap ---
sns.heatmap(
    D, annot=True, fmt='.3f', cmap='viridis',
    xticklabels=labels, yticklabels=labels,
    ax=axes[0], linewidths=0.5,
)
axes[0].set_title('SL Divergence Distance Matrix', fontsize=13)
axes[0].tick_params(axis='x', rotation=30)

# --- Dendrogram ---
# squareform converts the symmetric matrix to condensed form for linkage
condensed = squareform(D, checks=False)
Z = linkage(condensed, method='average')
dendrogram(Z, labels=labels, ax=axes[1], leaf_font_size=12, color_threshold=0)
axes[1].set_title('Hierarchical Clustering (UPGMA)', fontsize=13)
axes[1].set_ylabel('SL Divergence')

plt.tight_layout()
plt.savefig('dna_phylogenetic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: examples/dna_phylogenetic.png')

## 4. Verify against known phylogeny

In [ ]:
import pandas as pd

df = pd.DataFrame(D, index=labels, columns=labels)

# Off-diagonal distances from Human
human_row = df.loc['Human'].drop('Human').sort_values()
print('Distances from Human (ascending):')
print(human_row.to_string())

print()
expected_order = ['Chimp', 'Gorilla', 'Orangutan', 'Mouse']
actual_order   = human_row.index.tolist()

if actual_order == expected_order:
    print('✓ Distance order matches known primate phylogeny.')
else:
    print(f'Actual order:   {actual_order}')
    print(f'Expected order: {expected_order}')
    print('Note: small deviations can occur due to SAE stochasticity — '
          'try increasing num_repeat for more stable estimates.')